# ACM — Flight Price Prediction
**Lead University · Minería de Datos · Tarea 4**

Análisis de Correspondencias Múltiples sobre datos de vuelos comerciales.
Objetivo: explorar asociaciones entre variables categóricas del servicio aéreo (aerolínea, ruta, clase, horarios) y detectar perfiles de vuelos.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prince import MCA
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

RNG = 42
print('Librerías cargadas.')

---
## Parte 1 — Preparación de datos

### 1. Carga y exploración

In [ ]:
df_raw = pd.read_csv('datos/datos_vuelos_comerciales.csv', index_col=0)
print(f'Registros: {df_raw.shape[0]:,} | Variables: {df_raw.shape[1]}')
print()
print(df_raw.dtypes)
print()
df_raw.head()

In [ ]:
print('Valores faltantes:')
print(df_raw.isnull().sum())
print()
print('Valores únicos por variable categórica:')
for c in df_raw.select_dtypes(include='object').columns:
    print(f'  {c}: {df_raw[c].nunique()} -> {df_raw[c].unique().tolist()}')

### 2. Limpieza

El dataset no presenta valores faltantes. Se elimina la columna `flight` (identificador de vuelo, no aporta información categórica interpretable) y las variables numéricas se discretizan en el paso siguiente.

In [ ]:
df = df_raw.drop(columns=['flight']).copy()
print(f'Registros: {len(df):,} | Variables: {df.shape[1]}')

### 3-4. Selección de variables categóricas y discretización

Se seleccionan las variables categóricas naturales: `airline`, `source_city`, `departure_time`, `stops`, `arrival_time`, `destination_city`, `class`.

Adicionalmente se discretizan las variables numéricas `duration`, `days_left` y `price` en cuartiles para incorporarlas al ACM como factores que enriquecen el análisis.

In [ ]:
cat_cols = ['airline', 'source_city', 'departure_time', 'stops',
            'arrival_time', 'destination_city', 'class']

# Discretización de numéricas
df['duration_cat'] = pd.qcut(df['duration'], q=4, labels=['corta', 'media', 'larga', 'muy_larga'])
df['days_left_cat'] = pd.cut(df['days_left'], bins=[0, 7, 15, 30, 50],
                              labels=['ultimo_momento', 'pronto', 'anticipado', 'muy_anticipado'])
df['price_cat'] = pd.qcut(df['price'], q=4, labels=['economico', 'moderado', 'alto', 'premium'])

cat_cols_full = cat_cols + ['duration_cat', 'days_left_cat', 'price_cat']

df_acm = df[cat_cols_full].dropna().astype(str).reset_index(drop=True)
print(f'Registros para ACM: {len(df_acm):,} | Variables: {len(cat_cols_full)}')
print()
print('Modalidades por variable:')
for c in cat_cols_full:
    print(f'  {c}: {df_acm[c].nunique()} categorías')
print(f'\nTotal de modalidades: {sum(df_acm[c].nunique() for c in cat_cols_full)}')

El dataset tiene 300K registros, lo cual hace el ACM computacionalmente costoso para visualización. Se toma una muestra aleatoria estratificada para el ajuste.

In [ ]:
N_SAMPLE = 15000
df_sample = df_acm.sample(n=N_SAMPLE, random_state=RNG).reset_index(drop=True)
print(f'Muestra para ACM: {len(df_sample):,} registros')

---
## Parte 2 — Aplicación del ACM

### 6. ¿Qué es el ACM?

El **Análisis de Correspondencias Múltiples (ACM)** es una técnica de reducción de dimensionalidad diseñada para variables categóricas. Generaliza el Análisis de Correspondencias Simple (ACS) a más de dos variables.

**Objetivo:** Representar individuos y modalidades (categorías) en un espacio de pocas dimensiones, de modo que las distancias reflejen las asociaciones entre ellos (medidas vía chi-cuadrado). Puntos cercanos = perfiles similares; puntos opuestos = perfiles contrastantes.

**¿Por qué es apropiado para este dataset?** Todas las variables seleccionadas son categóricas nominales (aerolínea, ciudad, horario, clase). El PCA requiere variables numéricas y asume relaciones lineales; el ACM respeta la naturaleza categórica de los datos y detecta asociaciones no lineales entre modalidades.

### 5. Ajuste del ACM

In [ ]:
N_COMP = 10
mca = MCA(n_components=N_COMP, random_state=RNG)
mca.fit(df_sample)

print(f'Inercia total: {mca.total_inertia_:.4f}')
print(f'Componentes ajustados: {N_COMP}')

### 7. Tabla de inercia explicada por componente

In [ ]:
eigenvalues = mca.eigenvalues_
inercia_pct = 100 * eigenvalues / mca.total_inertia_
inercia_acum = np.cumsum(inercia_pct)

tabla_inercia = pd.DataFrame({
    'Dimensión': range(1, N_COMP + 1),
    'Autovalor': eigenvalues,
    '% Inercia': inercia_pct,
    '% Acumulada': inercia_acum
}).round(4)

display(tabla_inercia)

### 8. Scree plot (inercia acumulada)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(1, N_COMP + 1)

ax.bar(x, inercia_pct, color='steelblue', alpha=0.85, label='% inercia')
ax2 = ax.twinx()
ax2.plot(x, inercia_acum, 'o-', color='darkred', lw=2, markersize=5, label='% acumulada')
ax2.axhline(80, ls='--', color='gray', lw=0.8, label='80%')

ax.set_xlabel('Dimensión')
ax.set_ylabel('% Inercia')
ax2.set_ylabel('% Inercia acumulada')
ax.set_xticks(x)
ax.set_title('Scree Plot — ACM Vuelos')
ax2.legend(loc='center right')
plt.tight_layout()
plt.show()

### 9. Elección de componentes

**Nota sobre la inercia en ACM:** A diferencia del PCA donde los primeros componentes suelen acumular >80% rápidamente, en ACM la inercia se distribuye de forma más uniforme entre muchas dimensiones. Esto es inherente al método: la inercia total se divide entre (total_modalidades - num_variables) dimensiones posibles. Por lo tanto, porcentajes de inercia bajos (5-15% por dimensión) son normales y no implican falta de estructura.

Se seleccionan las **primeras 3 dimensiones** como adecuadas para la interpretación, ya que:
1. Muestran el mayor salto relativo respecto a las siguientes (criterio del codo).
2. Son las que concentran la mayor parte de la estructura interpretable.
3. Permiten una representación visual manejable (plano 2D + tercera dimensión como complemento).

---
## Parte 3 — Interpretación del espacio factorial

### 10. Biplot del ACM (individuos + categorías)

In [ ]:
coords_ind = mca.row_coordinates(df_sample)
coords_mod = mca.column_coordinates(df_sample)

fig, ax = plt.subplots(figsize=(12, 9))

# Individuos (muestra aleatoria para no saturar)
n_plot_ind = 3000
idx_plot = np.random.default_rng(RNG).choice(len(coords_ind), size=n_plot_ind, replace=False)
ax.scatter(coords_ind.iloc[idx_plot, 0], coords_ind.iloc[idx_plot, 1],
           s=8, alpha=0.2, c='steelblue', linewidths=0, label='Individuos')

# Modalidades
ax.scatter(coords_mod.iloc[:, 0], coords_mod.iloc[:, 1],
           s=60, marker='s', c='darkred', edgecolors='white', linewidths=0.5,
           zorder=3, label='Modalidades')

for idx, row in coords_mod.iterrows():
    ax.annotate(idx, (row.iloc[0], row.iloc[1]), fontsize=7, color='darkred',
                ha='left', va='bottom')

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Biplot ACM — Vuelos (individuos + modalidades)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 11. Interpretación visual del biplot

In [ ]:
# Mapa solo de modalidades (más legible para interpretar asociaciones)
fig, ax = plt.subplots(figsize=(13, 10))

# Colorear por variable de origen
colors = plt.cm.tab10(np.linspace(0, 1, len(cat_cols_full)))
color_map = {col: colors[i] for i, col in enumerate(cat_cols_full)}

for var in cat_cols_full:
    mask = coords_mod.index.str.startswith(var + '_') | (coords_mod.index.str.split('_').str[0] == var)
    # Filtrar las modalidades de esta variable
    var_mods = [idx for idx in coords_mod.index if idx.startswith(var)]
    if var_mods:
        subset = coords_mod.loc[var_mods]
        ax.scatter(subset.iloc[:, 0], subset.iloc[:, 1],
                   s=80, marker='s', c=[color_map[var]], edgecolors='white',
                   linewidths=0.5, zorder=3, label=var)
        for idx, row in subset.iterrows():
            ax.annotate(idx, (row.iloc[0], row.iloc[1]), fontsize=7,
                        ha='left', va='bottom')

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Mapa de modalidades — ACM Vuelos')
ax.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**Interpretación del biplot:**

*(Esta sección se completa tras ejecutar el notebook y observar las posiciones reales de las modalidades en el plano.)*

Puntos a observar:
- **Categorías cercanas:** Modalidades que aparecen próximas indican perfiles de vuelo que tienden a co-ocurrir (mismos individuos eligen combinaciones similares).
- **Asociaciones esperadas:** Es probable que `class_Business` se asocie con `price_premium` y aerolíneas de servicio completo (Vistara, Air India); mientras que `class_Economy` se agrupe con precios económicos y aerolíneas low-cost (SpiceJet, AirAsia).
- **Patrones de ruta:** Las ciudades de origen y destino que forman rutas frecuentes deben aparecer próximas.
- **Cerca del origen:** Modalidades muy frecuentes (Economy, zero stops) tienden al centro por ser el perfil promedio.

### 12. Contribuciones a las primeras dimensiones

In [ ]:
contrib = mca.column_contributions_
contrib_top = contrib.iloc[:, :3].copy()
contrib_top.columns = ['Dim 1 (%)', 'Dim 2 (%)', 'Dim 3 (%)']
contrib_top = (contrib_top * 100).round(2)

print('Top 15 modalidades por contribución a Dim 1:')
display(contrib_top.sort_values('Dim 1 (%)', ascending=False).head(15))
print()
print('Top 15 modalidades por contribución a Dim 2:')
display(contrib_top.sort_values('Dim 2 (%)', ascending=False).head(15))

In [ ]:
# Contribución agregada por variable (sumando modalidades)
contrib_var = pd.DataFrame(index=cat_cols_full, columns=['Dim 1 (%)', 'Dim 2 (%)'])
for var in cat_cols_full:
    var_mods = [idx for idx in contrib.index if idx.startswith(var)]
    if var_mods:
        contrib_var.loc[var, 'Dim 1 (%)'] = (contrib.loc[var_mods, 0] * 100).sum()
        contrib_var.loc[var, 'Dim 2 (%)'] = (contrib.loc[var_mods, 1] * 100).sum()

contrib_var = contrib_var.astype(float).round(2)
print('Contribución agregada por variable:')
display(contrib_var.sort_values('Dim 1 (%)', ascending=False))

### 13. Perfiles similares en el espacio reducido

In [ ]:
# Colorear individuos por clase para ver si el ACM separa perfiles
fig, ax = plt.subplots(figsize=(10, 7))

palette = {'Economy': 'steelblue', 'Business': 'coral'}
for clase, color in palette.items():
    mask = df_sample['class'] == clase
    ax.scatter(coords_ind.loc[mask, 0], coords_ind.loc[mask, 1],
               s=8, alpha=0.3, c=color, linewidths=0, label=clase)

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Plano ACM — Coloreado por clase')
ax.legend()
plt.tight_layout()
plt.show()

# Centroides por clase
print('Centroides por clase:')
coords_ind_class = coords_ind.iloc[:, :2].copy()
coords_ind_class['class'] = df_sample['class'].values
print(coords_ind_class.groupby('class')[[0, 1]].mean().round(3))

In [ ]:
# Perfiles por aerolínea
fig, ax = plt.subplots(figsize=(10, 7))

airlines = df_sample['airline'].unique()
colors_air = plt.cm.Set2(np.linspace(0, 1, len(airlines)))

for i, aero in enumerate(sorted(airlines)):
    mask = df_sample['airline'] == aero
    ax.scatter(coords_ind.loc[mask, 0], coords_ind.loc[mask, 1],
               s=10, alpha=0.3, c=[colors_air[i]], linewidths=0, label=aero)

ax.axhline(0, c='gray', lw=0.5)
ax.axvline(0, c='gray', lw=0.5)
ax.set_xlabel(f'Dim 1 ({inercia_pct[0]:.1f}% inercia)')
ax.set_ylabel(f'Dim 2 ({inercia_pct[1]:.1f}% inercia)')
ax.set_title('Plano ACM — Coloreado por aerolínea')
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

print('Centroides por aerolínea:')
coords_ind_air = coords_ind.iloc[:, :2].copy()
coords_ind_air['airline'] = df_sample['airline'].values
print(coords_ind_air.groupby('airline')[[0, 1]].mean().round(3))

**Análisis de perfiles:**

*(Se completa tras ejecutar — observar si el ACM logra separar grupos distinguibles.)*

Se espera observar al menos dos perfiles principales:
1. **Perfil business:** Vuelos de aerolíneas premium (Vistara, Air India), precios altos, posiblemente más escalas o duraciones largas.
2. **Perfil economy/low-cost:** Aerolíneas económicas (SpiceJet, AirAsia, Indigo), vuelos directos, precios bajos, rutas cortas.

### Cos² — Calidad de representación

In [ ]:
cos2 = mca.column_cosine_similarities(df_sample)
cos2_plano = cos2.iloc[:, :2].copy()
cos2_plano.columns = ['cos² Dim1', 'cos² Dim2']
cos2_plano['cos² total (1+2)'] = cos2_plano.sum(axis=1)

print('Modalidades mejor representadas en el plano (top 15):')
display(cos2_plano.sort_values('cos² total (1+2)', ascending=False).head(15).round(4))